In [70]:
# Sam Brown
# Sam_brown@mines.edu
# June 20
# Goal: Use LSTM neural nets to capture and leverage long and short term patterns in the tidal modulation

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split, TensorDataset
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

df = pd.read_csv("09-18.csv", parse_dates=["start_time"])
df = df.iloc[1:4564] # NANS UNKNOWN CHECK AGAIN

In [72]:
# Features and Target
X = df[['tide_h', 'tide_deriv', 'form_fac', 'time_since', 'high_t_evt', 'tide_height']]
y = df['slip_size']

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [125]:
SEQ_LEN = 25 # Sequence of "memory"
batch_size = 32

sequences = []
targets = []

# We Want to create rolling sequences of 10
for i in range(len(X_scaled) - SEQ_LEN):
    seq = X_scaled[i:i+SEQ_LEN]
    target = y.iloc[i + SEQ_LEN]  # slip_size AFTER the sequence
    sequences.append(seq)
    targets.append(target)

#Tensors
X_seq = torch.tensor(np.array(sequences), dtype=torch.float32)
y_seq = torch.tensor(np.array(targets), dtype=torch.float32).unsqueeze(1)

print("X_seq shape:", X_seq.shape)  # [num_samples, seq_len, 6]
print("y_seq shape:", y_seq.shape)  # [num_samples, 1]

X_seq shape: torch.Size([4538, 25, 6])
y_seq shape: torch.Size([4538, 1])


In [127]:
# Wrap to Tensor Dataset
dataset = TensorDataset(X_seq, y_seq)

# train test 80 20
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_set, test_set = random_split(dataset, [train_size, test_size])

# Create DataLoaders
train_loader = DataLoader(train_set, batch_size, shuffle=True # Splits into mini-batches
test_loader = DataLoader(test_set, batch_size, shuffle=False)

In [129]:
# Model
class SlipLSTM(nn.Module):
    def __init__(self, input_size=6, hidden_size=64, num_layers=1):
        super(SlipLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size=input_size,
                            hidden_size=hidden_size,
                            num_layers=num_layers,
                            batch_first=True)

        self.fc1 = nn.Linear(hidden_size, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32,1)

    def forward(self, x):
        out, _ = self.lstm(x)       # out: [batch, seq_len, hidden_size]
        out = out[:, -1, :]         # take output at last time step
        out = self.fc1(out)         # linear layer 1
        out = self.relu(out)        # ReLU activation
        out = self.fc2(out)         # final output layer
        return out

In [130]:
model = SlipLSTM(input_size=6, hidden_size=64, num_layers=1)

# Loss and optimizer
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [133]:
num_epochs = 30

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for batch_X, batch_y in train_loader: # Mini- batch loop
        optimizer.zero_grad()
        preds = model(batch_X)
        loss = loss_fn(preds, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

Epoch 1/30, Loss: 1.0393
Epoch 2/30, Loss: 0.9174
Epoch 3/30, Loss: 0.8734
Epoch 4/30, Loss: 0.8334
Epoch 5/30, Loss: 0.8124
Epoch 6/30, Loss: 0.7908
Epoch 7/30, Loss: 0.7641
Epoch 8/30, Loss: 0.7202
Epoch 9/30, Loss: 0.7007
Epoch 10/30, Loss: 0.6721
Epoch 11/30, Loss: 0.6566
Epoch 12/30, Loss: 0.6244
Epoch 13/30, Loss: 0.6070
Epoch 14/30, Loss: 0.5935
Epoch 15/30, Loss: 0.5811
Epoch 16/30, Loss: 0.5762
Epoch 17/30, Loss: 0.5545
Epoch 18/30, Loss: 0.5523
Epoch 19/30, Loss: 0.5316
Epoch 20/30, Loss: 0.5171
Epoch 21/30, Loss: 0.5111
Epoch 22/30, Loss: 0.4988
Epoch 23/30, Loss: 0.4888
Epoch 24/30, Loss: 0.4724
Epoch 25/30, Loss: 0.4723
Epoch 26/30, Loss: 0.4543
Epoch 27/30, Loss: 0.4500
Epoch 28/30, Loss: 0.4363
Epoch 29/30, Loss: 0.4260
Epoch 30/30, Loss: 0.4213


In [135]:
model.eval()

preds, trues = [], []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        pred = model(batch_X)
        preds.append(pred.numpy())
        trues.append(batch_y.numpy())

preds = np.vstack(preds)
trues = np.vstack(trues)

mse = mean_squared_error(trues, preds)
print(f"Test MSE: {mse:.4f}")

Test MSE: 0.6773


In [148]:

r2 = r2_score(trues, preds)
print("R² score:", r2)

R² score: 0.4129123091697693


In [ ]:
# MSE: .7995: 30 epochs, seq of 10, learning rate .001, adam optimizer, batch size 16 
# MSE: .5932: 30 epoches, seq of 25, bach size 16
# MSE: .6773: 30 epochs, seq of 25, batch size 32